# Model Comparison for GLCM Skin Disease Classification

This notebook compares several classical machine learning models on the same GLCM feature set used by the Flask app and training script.

The goal is to answer a simple question: **which model gives the best accuracy on the SkinDisease dataset?**

## What this notebook does

1. Loads the train and test images from `data/SkinDisease/SkinDisease`.
2. Extracts GLCM texture features from every image.
3. Trains and evaluates multiple models.
4. Compares the results in a table and a bar chart.

## Models included

- Random Forest
- Support Vector Machine (SVM)
- Logistic Regression
- k-Nearest Neighbors (KNN)
- Decision Tree

You can add more models later if you want to experiment further.

In [9]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from config import DATASET_ROOT
from features import GLCMConfig, extract_glcm_features, describe_feature_vector

plt.style.use('seaborn-v0_8-whitegrid')

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

@dataclass(frozen=True)
class Sample:
    path: Path
    label: str
    split: str

print('Dataset root:', DATASET_ROOT)
print('Feature count:', len(describe_feature_vector()))

Dataset root: /Users/muhghifari/Documents/TA Pengolahan Citra/data/SkinDisease/SkinDisease
Feature count: 28


In [10]:
def discover_samples(split_dir: Path, split_name: str) -> list[Sample]:
    if not split_dir.is_dir():
        raise FileNotFoundError(f'Missing dataset directory: {split_dir}')

    samples: list[Sample] = []
    for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
        for path in sorted(class_dir.rglob('*')):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                samples.append(Sample(path=path, label=class_dir.name, split=split_name))

    if not samples:
        raise ValueError(f'No images found in {split_dir}.')
    return samples


def limit_samples(samples: list[Sample], max_images_per_class: int | None, seed: int = 42) -> list[Sample]:
    if max_images_per_class is None:
        return samples

    grouped: dict[str, list[Sample]] = defaultdict(list)
    for sample in samples:
        grouped[sample.label].append(sample)

    rng = random.Random(seed)
    limited: list[Sample] = []
    for label in sorted(grouped):
        bucket = grouped[label]
        if len(bucket) > max_images_per_class:
            bucket = rng.sample(bucket, max_images_per_class)
        limited.extend(sorted(bucket, key=lambda item: item.path.name))
    return limited


def build_matrix(samples: list[Sample], image_size: int = 128, levels: int = 16) -> tuple[np.ndarray, np.ndarray]:
    features: list[np.ndarray] = []
    labels: list[str] = []
    config = GLCMConfig(image_size=image_size, levels=levels)

    for sample in samples:
        try:
            features.append(extract_glcm_features(sample.path, config=config))
            labels.append(sample.label)
        except ValueError as exc:
            print(f'Skipping {sample.path}: {exc}')

    if not features:
        raise ValueError('No valid images could be processed.')

    return np.vstack(features), np.asarray(labels)


SEED = 42
IMAGE_SIZE = 128
LEVELS = 16
MAX_IMAGES_PER_CLASS = None  # set to a small number like 5 for a quick smoke test

train_samples = limit_samples(discover_samples(DATASET_ROOT / 'train', 'train'), max_images_per_class=MAX_IMAGES_PER_CLASS, seed=SEED)
test_samples = limit_samples(discover_samples(DATASET_ROOT / 'test', 'test'), max_images_per_class=MAX_IMAGES_PER_CLASS, seed=SEED)

x_train, y_train = build_matrix(train_samples, image_size=IMAGE_SIZE, levels=LEVELS)
x_test, y_test = build_matrix(test_samples, image_size=IMAGE_SIZE, levels=LEVELS)

print('Train shape:', x_train.shape, y_train.shape)
print('Test shape:', x_test.shape, y_test.shape)
print('Classes:', sorted(set(y_train) | set(y_test)))

Train shape: (13898, 28) (13898,)
Test shape: (1546, 28) (1546,)
Classes: [np.str_('Acne'), np.str_('Actinic_Keratosis'), np.str_('Benign_tumors'), np.str_('Bullous'), np.str_('Candidiasis'), np.str_('DrugEruption'), np.str_('Eczema'), np.str_('Infestations_Bites'), np.str_('Lichen'), np.str_('Lupus'), np.str_('Moles'), np.str_('Psoriasis'), np.str_('Rosacea'), np.str_('Seborrh_Keratoses'), np.str_('SkinCancer'), np.str_('Sun_Sunlight_Damage'), np.str_('Tinea'), np.str_('Unknown_Normal'), np.str_('Vascular_Tumors'), np.str_('Vasculitis'), np.str_('Vitiligo'), np.str_('Warts')]


In [11]:
models = {
    'Dummy (most frequent)': DummyClassifier(strategy='most_frequent'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=7)),
    ]),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', multi_class='auto')),
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)),
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight='balanced_subsample',
        n_jobs=-1,
    ),
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)
    result = {
        'model': name,
        'accuracy': accuracy_score(y_test, predictions),
        'precision': precision_score(y_test, predictions, average='weighted', zero_division=0),
        'recall': recall_score(y_test, predictions, average='weighted', zero_division=0),
        'f1_score': f1_score(y_test, predictions, average='weighted', zero_division=0),
    }
    results.append(result)
    fitted_models[name] = model

results_sorted = sorted(results, key=lambda item: item['accuracy'], reverse=True)
for item in results_sorted:
    print(
        f"{item['model']:<24} | accuracy={item['accuracy']:.4f} | precision={item['precision']:.4f} | recall={item['recall']:.4f} | f1={item['f1_score']:.4f}"
    )

/Users/muhghifari/Documents/TA Pengolahan Citra/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/muhghifari/Documents/TA Pengolahan Citra/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Random Forest            | accuracy=0.3137 | precision=0.3022 | recall=0.3137 | f1=0.2999
Decision Tree            | accuracy=0.2296 | precision=0.2372 | recall=0.2296 | f1=0.2323
KNN                      | accuracy=0.2283 | precision=0.2259 | recall=0.2283 | f1=0.2200
Logistic Regression      | accuracy=0.2257 | precision=0.2399 | recall=0.2257 | f1=0.2128
SVM (RBF)                | accuracy=0.2147 | precision=0.2531 | recall=0.2147 | f1=0.2009
Dummy (most frequent)    | accuracy=0.1223 | precision=0.0149 | recall=0.1223 | f1=0.0266


In [12]:
print('Model comparison complete. The results are shown above in sorted order.')
print('Use the best model based on accuracy and then inspect precision/recall if classes are imbalanced.')

Model comparison complete. The results are shown above in sorted order.
Use the best model based on accuracy and then inspect precision/recall if classes are imbalanced.


In [14]:
import io
from PIL import Image

# Create chart using PIL to avoid matplotlib recursion bug
ordered = sorted(results, key=lambda item: item['accuracy'], reverse=True)
model_names = [item['model'] for item in ordered]
accuracies = [item['accuracy'] * 100 for item in ordered]

# Create a simple text summary instead of a chart
print("\n=== Model Accuracy Comparison ===\n")
for model_name, acc in zip(model_names, accuracies):
    bar_length = int(acc / 5)  # 20 chars for 100%
    bar = '█' * bar_length + '░' * (20 - bar_length)
    color_mark = '⭐' if model_name == 'Random Forest' else '  '
    print(f"{color_mark} {model_name:25} | {bar} | {acc:5.1f}%")

print("\n✓ Model comparison complete. Random Forest is highlighted if selected.")



=== Model Accuracy Comparison ===

⭐ Random Forest             | ██████░░░░░░░░░░░░░░ |  31.4%
   Decision Tree             | ████░░░░░░░░░░░░░░░░ |  23.0%
   KNN                       | ████░░░░░░░░░░░░░░░░ |  22.8%
   Logistic Regression       | ████░░░░░░░░░░░░░░░░ |  22.6%
   SVM (RBF)                 | ████░░░░░░░░░░░░░░░░ |  21.5%
   Dummy (most frequent)     | ██░░░░░░░░░░░░░░░░░░ |  12.2%

✓ Model comparison complete. Random Forest is highlighted if selected.


In [15]:
best_model_name = max(results, key=lambda item: item['accuracy'])['model']
best_model = fitted_models[best_model_name]
best_predictions = best_model.predict(x_test)

print('Best model:', best_model_name)
print('\n' + classification_report(y_test, best_predictions, zero_division=0))

label_order = sorted(set(y_test))
matrix = confusion_matrix(y_test, best_predictions, labels=label_order)

print(f"\nConfusion Matrix - {best_model_name}")
print(f"Shape: {matrix.shape[0]} classes × {matrix.shape[1]} predictions")
print(f"Total test samples: {matrix.sum()}")
print(f"Diagonal (correct predictions): {matrix.trace()}")


Best model: Random Forest

                     precision    recall  f1-score   support

               Acne       0.18      0.26      0.21        65
  Actinic_Keratosis       0.18      0.19      0.19        83
      Benign_tumors       0.24      0.35      0.28       121
            Bullous       0.28      0.15      0.19        55
        Candidiasis       0.24      0.19      0.21        27
       DrugEruption       0.32      0.26      0.29        61
             Eczema       0.32      0.46      0.37       112
 Infestations_Bites       0.25      0.17      0.20        60
             Lichen       0.17      0.11      0.14        61
              Lupus       0.33      0.12      0.17        34
              Moles       0.24      0.20      0.22        40
          Psoriasis       0.23      0.31      0.26        88
            Rosacea       0.26      0.21      0.24        28
  Seborrh_Keratoses       0.30      0.24      0.26        51
         SkinCancer       0.15      0.14      0.15       

## How to interpret the results

- The model with the highest accuracy is usually the best first choice for your report.
- Precision and recall are useful if some classes are harder to detect than others.
- The confusion matrix shows which classes are commonly confused.

### Good default for the final project

If `Random Forest` performs best or near the best, it is a strong choice because it is easy to explain, fast to train, and already matches the Flask app.

If another model performs better, you can switch the app to use that model later, but make sure to keep the same feature extraction settings.